# Time Series Forecasting -- Daily Female Births in California (1959)

---

## Problem Statement and Context

Time series forecasting is fundamental to decision-making in domains ranging from public health resource planning and hospital staffing to demographic research and government policy formulation. Accurate predictions of daily birth counts allow healthcare systems to allocate resources (delivery rooms, nursing staff, neonatal equipment) efficiently and anticipate periods of high demand.

This notebook analyzes a classic time series dataset recording the daily total number of female births in California throughout 1959. The dataset comprises 365 observations -- one per day -- providing a full year of daily granularity. The objective is to build, evaluate, and compare multiple forecasting approaches to predict future birth counts.

**What is a time series?**

A time series is a sequence of data points collected at successive, equally spaced intervals over time. Unlike cross-sectional data, time series data exhibits temporal dependencies where past values influence future outcomes. Key components include:

- Level: The baseline average value of the series.
- Trend: A long-term increase or decrease in the data. An uptrend forms higher highs and higher lows, while a downtrend forms lower highs and lower lows.
- Seasonality: Regular, repeating patterns tied to calendar periods (weekly, monthly, quarterly cycles).
- Noise: Random, irregular fluctuations that cannot be attributed to trend or seasonality.

**Forecasting Approaches**

Traditional statistical methods (ARIMA, Exponential Smoothing) model the autocorrelation structure directly. Machine learning approaches reframe the problem as supervised learning using lagged features. This notebook implements both paradigms and compares their performance.

**Notebook Coverage**

- Data loading and temporal structure analysis
- Decomposition into trend, seasonal, and residual components
- Stationarity testing (ADF test) and differencing
- Autocorrelation and partial autocorrelation analysis
- Classical methods: Moving Average, Exponential Smoothing, ARIMA, SARIMA, Auto ARIMA
- Machine learning methods: Random Forest, Gradient Boosting, XGBoost with lagged features
- Ensemble combination of forecasts
- Evaluation using RMSE, MAE, and MAPE

## Library Imports

In [ ]:
import subprocess
import sys

# Ensure compatible versions of scipy and statsmodels are installed
# This resolves the scipy._lib._util ImportError
for pkg in ['scipy', 'statsmodels', 'xgboost', 'pmdarima']:
    try:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '--upgrade', pkg, '-q'],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
    except subprocess.CalledProcessError:
        # Some environments require --break-system-packages
        try:
            subprocess.check_call(
                [sys.executable, '-m', 'pip', 'install', '--upgrade', pkg,
                 '--break-system-packages', '-q'],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
            )
        except Exception:
            pass  # Package may already be at a compatible version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

plt.rcParams['figure.dpi'] = 100
print("All libraries loaded.")

## Data Loading and Temporal Structure

In [ ]:
# Load with date parsing and set daily frequency
birth_data = pd.read_csv('daily-total-female-births-CA.csv', parse_dates=['date'], index_col='date')
birth_data.index.freq = 'D'  # Set explicit daily frequency to avoid statsmodels warnings

print("Dataset shape:", birth_data.shape)
print("Date range: {} to {}".format(birth_data.index.min().date(), birth_data.index.max().date()))
print("Frequency: Daily ({} observations)".format(len(birth_data)))
print()
birth_data.head(10)

In [ ]:
birth_data.describe()

In [ ]:
# Check for missing dates and values
print("Missing values:", birth_data.isnull().sum().values[0])
print("Date continuity check:")
date_range = pd.date_range(start=birth_data.index.min(), end=birth_data.index.max(), freq='D')
missing_dates = date_range.difference(birth_data.index)
print("  Missing dates: {}".format(len(missing_dates)))

## Visual Exploration of the Series

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# Raw series
axes[0].plot(birth_data.index, birth_data['births'], color='steelblue', linewidth=0.8)
axes[0].set_title('Daily Female Births -- California 1959', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Births')
axes[0].axhline(birth_data['births'].mean(), color='red', linestyle='--', alpha=0.6,
                label='Mean: {:.1f}'.format(birth_data['births'].mean()))
axes[0].legend()

# Rolling statistics (smoothing)
rolling_mean = birth_data['births'].rolling(window=7).mean()
rolling_std = birth_data['births'].rolling(window=7).std()
axes[1].plot(birth_data.index, birth_data['births'], color='steelblue', alpha=0.4, linewidth=0.7, label='Original')
axes[1].plot(rolling_mean.index, rolling_mean, color='red', linewidth=1.5, label='7-day Rolling Mean')
axes[1].plot(rolling_std.index, rolling_std, color='green', linewidth=1.5, label='7-day Rolling Std')
axes[1].set_title('Rolling Mean and Standard Deviation (window=7)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Value')
axes[1].legend()

# 30-day moving average
ma_30 = birth_data['births'].rolling(window=30).mean()
axes[2].plot(birth_data.index, birth_data['births'], color='steelblue', alpha=0.3, linewidth=0.7, label='Original')
axes[2].plot(ma_30.index, ma_30, color='darkred', linewidth=2, label='30-day Moving Average')
axes[2].set_title('30-Day Moving Average -- Reveals Monthly Trends', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Number of Births')
axes[2].legend()

plt.tight_layout()
plt.show()

### Monthly and Weekly Patterns

In [ ]:
birth_data_copy = birth_data.copy()
birth_data_copy['month'] = birth_data_copy.index.month
birth_data_copy['day_of_week'] = birth_data_copy.index.dayofweek
birth_data_copy['week'] = birth_data_copy.index.isocalendar().week.values.astype(int)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Monthly aggregation
monthly = birth_data_copy.groupby('month')['births'].mean()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[0].bar(month_names, monthly.values, color='steelblue', edgecolor='black')
axes[0].set_title('Average Daily Births by Month', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Average Births')
for i, v in enumerate(monthly.values):
    axes[0].text(i, v + 0.3, '{:.1f}'.format(v), ha='center', fontsize=8)

# Day of week
dow_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow = birth_data_copy.groupby('day_of_week')['births'].mean()
axes[1].bar(dow_names, dow.values, color='coral', edgecolor='black')
axes[1].set_title('Average Daily Births by Day of Week', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Average Births')
for i, v in enumerate(dow.values):
    axes[1].text(i, v + 0.3, '{:.1f}'.format(v), ha='center', fontsize=8)

plt.tight_layout()
plt.show()

### Distribution of Birth Counts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(birth_data['births'], bins=25, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(birth_data['births'].mean(), color='red', linestyle='--', label='Mean')
axes[0].axvline(birth_data['births'].median(), color='green', linestyle='--', label='Median')
axes[0].set_title('Histogram of Daily Births', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Births')
axes[0].set_ylabel('Frequency')
axes[0].legend()

sm.qqplot(birth_data['births'], line='s', ax=axes[1])
axes[1].set_title('Q-Q Plot', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## Stationarity Analysis

A stationary time series has constant statistical properties (mean, variance, autocorrelation) over time. Most forecasting models require stationarity. We use the Augmented Dickey-Fuller (ADF) test where the null hypothesis is that the series has a unit root (non-stationary).

In [ ]:
def run_adf_test(series, series_name='Series'):
    result = adfuller(series.dropna(), autolag='AIC')
    print("ADF Test Results for: {}".format(series_name))
    print("-" * 45)
    print("  Test Statistic:   {:.6f}".format(result[0]))
    print("  P-value:          {:.6f}".format(result[1]))
    print("  Lags Used:        {}".format(result[2]))
    print("  Observations:     {}".format(result[3]))
    print("  Critical Values:")
    for key, val in result[4].items():
        print("    {}: {:.4f}".format(key, val))
    if result[1] < 0.05:
        print("  Conclusion: Series IS stationary (reject H0)")
    else:
        print("  Conclusion: Series is NOT stationary (fail to reject H0)")
    print()
    return result[1]

# Test original series
pval = run_adf_test(birth_data['births'], 'Original Series')

# Test first difference
diff_1 = birth_data['births'].diff().dropna()
pval_diff = run_adf_test(diff_1, 'First Difference')

## Autocorrelation Analysis

In [ ]:
# Durbin-Watson test for autocorrelation
dw = sm.stats.durbin_watson(birth_data['births'].values)
print("Durbin-Watson statistic: {:.4f}".format(dw))
print("  (Values near 2 indicate no autocorrelation)")
print("  (Values near 0 indicate positive autocorrelation)")
print("  (Values near 4 indicate negative autocorrelation)")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ACF and PACF for original series
plot_acf(birth_data['births'].values, lags=40, ax=axes[0][0])
axes[0][0].set_title('ACF -- Original Series', fontsize=12, fontweight='bold')

plot_pacf(birth_data['births'].values, lags=40, ax=axes[0][1])
axes[0][1].set_title('PACF -- Original Series', fontsize=12, fontweight='bold')

# ACF and PACF for differenced series
plot_acf(diff_1.values, lags=40, ax=axes[1][0])
axes[1][0].set_title('ACF -- First Difference', fontsize=12, fontweight='bold')

plot_pacf(diff_1.values, lags=40, ax=axes[1][1])
axes[1][1].set_title('PACF -- First Difference', fontsize=12, fontweight='bold')

plt.suptitle('Autocorrelation and Partial Autocorrelation Functions',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Time Series Decomposition

In [ ]:
# Additive decomposition with period = 7 (weekly cycle)
decomposition = seasonal_decompose(birth_data['births'], model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(15, 12))

decomposition.observed.plot(ax=axes[0], color='steelblue')
axes[0].set_title('Observed', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Births')

decomposition.trend.plot(ax=axes[1], color='red')
axes[1].set_title('Trend Component', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Births')

decomposition.seasonal.plot(ax=axes[2], color='green')
axes[2].set_title('Seasonal Component (period=7)', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Births')

decomposition.resid.plot(ax=axes[3], color='purple')
axes[3].set_title('Residual Component', fontsize=12, fontweight='bold')
axes[3].set_ylabel('Births')

plt.suptitle('Time Series Decomposition (Additive)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Train-Test Split

We hold out the last 45 days for testing, using the first 320 days for training. This temporal split preserves the sequential nature of the data.

In [ ]:
train = birth_data[:320].copy()
test = birth_data[320:].copy()

# Ensure frequency is preserved after slicing
train.index.freq = 'D'
test.index.freq = 'D'

print("Training period: {} to {} ({} days)".format(
    train.index[0].date(), train.index[-1].date(), len(train)))
print("Test period:     {} to {} ({} days)".format(
    test.index[0].date(), test.index[-1].date(), len(test)))

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(train.index, train['births'], color='steelblue', label='Training Data')
ax.plot(test.index, test['births'], color='coral', label='Test Data')
ax.axvline(test.index[0], color='black', linestyle='--', alpha=0.5)
ax.set_title('Train-Test Split', fontsize=13, fontweight='bold')
ax.set_ylabel('Births')
ax.legend()
plt.tight_layout()
plt.show()

## Evaluation Utilities

In [ ]:
def evaluate_forecast(actual, predicted, model_name):
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    print("{:30s}  RMSE={:.4f}  MAE={:.4f}  MAPE={:.2f}%".format(model_name, rmse, mae, mape))
    return {'Model': model_name, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

all_results = []

## Baseline: Naive and Moving Average Forecasts

In [ ]:
# Naive forecast: predict last known value
naive_pred = np.full(len(test), train['births'].iloc[-1])
res = evaluate_forecast(test['births'].values, naive_pred, 'Naive (Last Value)')
all_results.append(res)

# Mean forecast: predict training mean
mean_pred = np.full(len(test), train['births'].mean())
res = evaluate_forecast(test['births'].values, mean_pred, 'Mean Forecast')
all_results.append(res)

# Moving average forecast (use last 7 days average as forecast)
ma_pred = np.full(len(test), train['births'].iloc[-7:].mean())
res = evaluate_forecast(test['births'].values, ma_pred, 'Moving Average (7-day)')
all_results.append(res)

## Simple Exponential Smoothing

In [ ]:
ses_model = SimpleExpSmoothing(train['births']).fit(smoothing_level=0.3, optimized=False)
ses_forecast = ses_model.forecast(steps=len(test))
res = evaluate_forecast(test['births'].values, ses_forecast.values, 'Simple Exponential Smoothing')
all_results.append(res)

## Holt-Winters Exponential Smoothing

In [ ]:
hw_model = ExponentialSmoothing(
    train['births'], trend='add', seasonal='add', seasonal_periods=7
).fit()
hw_forecast = hw_model.forecast(steps=len(test))
res = evaluate_forecast(test['births'].values, hw_forecast.values, 'Holt-Winters (Additive)')
all_results.append(res)

## ARIMA Model

Based on ACF/PACF analysis, we fit an ARIMA model. The order (p, d, q) is selected considering the patterns observed in the autocorrelation plots.

In [ ]:
# Fit ARIMA(2,1,3) as suggested by the original project scripts
arima_model = ARIMA(train['births'], order=(2, 1, 3)).fit()

print("ARIMA(2,1,3) Summary:")
print("  AIC:  {:.2f}".format(arima_model.aic))
print("  BIC:  {:.2f}".format(arima_model.bic))

arima_forecast = arima_model.forecast(steps=len(test))
res = evaluate_forecast(test['births'].values, arima_forecast.values, 'ARIMA(2,1,3)')
all_results.append(res)

In [ ]:
# Try alternative ARIMA orders
for order in [(1,1,1), (2,1,2), (3,1,2), (1,1,3)]:
    try:
        m = ARIMA(train['births'], order=order).fit()
        fc = m.forecast(steps=len(test))
        r = evaluate_forecast(test['births'].values, fc.values,
                              'ARIMA{}'.format(order))
        all_results.append(r)
    except Exception as e:
        print("ARIMA{} failed: {}".format(order, str(e)[:50]))

## SARIMA (Seasonal ARIMA)

SARIMA extends ARIMA by incorporating seasonal differencing and seasonal AR/MA terms, which is appropriate given the weekly pattern observed in the decomposition.

In [ ]:
sarima_model = SARIMAX(
    train['births'],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7)
).fit(disp=False)

print("SARIMA(1,1,1)(1,1,1,7) Summary:")
print("  AIC:  {:.2f}".format(sarima_model.aic))

sarima_forecast = sarima_model.forecast(steps=len(test))
res = evaluate_forecast(test['births'].values, sarima_forecast.values, 'SARIMA(1,1,1)(1,1,1,7)')
all_results.append(res)

In [ ]:
# Try another SARIMA configuration
sarima2 = SARIMAX(
    train['births'],
    order=(2, 1, 2),
    seasonal_order=(1, 0, 1, 7)
).fit(disp=False)

sarima2_forecast = sarima2.forecast(steps=len(test))
res = evaluate_forecast(test['births'].values, sarima2_forecast.values, 'SARIMA(2,1,2)(1,0,1,7)')
all_results.append(res)

## Auto ARIMA (Automated Order Selection)

We use pmdarima for automatic model selection through stepwise AIC minimization.

In [ ]:
from pmdarima import auto_arima

auto_model = auto_arima(
    train['births'], seasonal=True, m=7,
    stepwise=True, suppress_warnings=True,
    trace=False, error_action='ignore'
)
print("Auto ARIMA selected order: {} x {}".format(auto_model.order, auto_model.seasonal_order))
print("AIC: {:.2f}".format(auto_model.aic()))

auto_forecast = auto_model.predict(n_periods=len(test))
res = evaluate_forecast(test['births'].values, auto_forecast, 'Auto ARIMA')
all_results.append(res)

## Machine Learning Approaches with Lagged Features

Reframing the time series as a supervised learning problem: we create lag features (previous day values) and use them as predictors. This allows us to leverage powerful tree-based ensemble models.

In [ ]:
# Create lag features
def create_lag_features(series, n_lags=14):
    df_lag = pd.DataFrame({'births': series.values}, index=series.index)
    for i in range(1, n_lags + 1):
        df_lag['lag_{}'.format(i)] = df_lag['births'].shift(i)
    # Rolling features
    df_lag['rolling_mean_7'] = df_lag['births'].shift(1).rolling(window=7).mean()
    df_lag['rolling_std_7'] = df_lag['births'].shift(1).rolling(window=7).std()
    df_lag['rolling_mean_14'] = df_lag['births'].shift(1).rolling(window=14).mean()
    # Day of week
    df_lag['day_of_week'] = df_lag.index.dayofweek
    df_lag['month'] = df_lag.index.month
    df_lag = df_lag.dropna()
    return df_lag

full_data = create_lag_features(birth_data['births'], n_lags=14)
print("Feature matrix shape:", full_data.shape)
print("Features:", [c for c in full_data.columns if c != 'births'])
full_data.head()

In [ ]:
# Split based on the same date boundary
train_ml = full_data[full_data.index < test.index[0]]
test_ml = full_data[full_data.index >= test.index[0]]

feature_cols_ml = [c for c in full_data.columns if c != 'births']
X_train_ml = train_ml[feature_cols_ml]
y_train_ml = train_ml['births']
X_test_ml = test_ml[feature_cols_ml]
y_test_ml = test_ml['births']

print("ML Training: {} samples".format(len(X_train_ml)))
print("ML Test:     {} samples".format(len(X_test_ml)))

In [ ]:
# Train multiple ML models
ml_models = {
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=10,
                                            random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                                    learning_rate=0.05, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05,
                             random_state=42, n_jobs=-1),
}

ml_predictions = {}

print("Machine Learning Model Results")
print("=" * 65)

for name, model in ml_models.items():
    model.fit(X_train_ml, y_train_ml)
    pred = model.predict(X_test_ml)
    ml_predictions[name] = pred
    res = evaluate_forecast(y_test_ml.values, pred, name)
    all_results.append(res)

In [ ]:
# Feature importance from Random Forest
rf_model = ml_models['Random Forest']
importances = rf_model.feature_importances_
imp_df = pd.DataFrame({'Feature': feature_cols_ml, 'Importance': importances})
imp_df = imp_df.sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(imp_df['Feature'], imp_df['Importance'], color='steelblue', edgecolor='black')
ax.set_title('Random Forest Feature Importance (Time Series)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## Ensemble Combination of Forecasts

Combining forecasts from multiple models often produces more accurate and robust predictions than any single model.

In [ ]:
# Average ensemble of ML models
ml_avg = np.mean([ml_predictions['Random Forest'],
                   ml_predictions['Gradient Boosting'],
                   ml_predictions['XGBoost']], axis=0)
res = evaluate_forecast(y_test_ml.values, ml_avg, 'ML Ensemble (Average)')
all_results.append(res)

## Forecast Visualization

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 14))

# Statistical models
axes[0].plot(test.index, test['births'].values, 'k-', linewidth=2, label='Actual', alpha=0.8)
axes[0].plot(test.index, arima_forecast.values, '--', label='ARIMA(2,1,3)', linewidth=1.2)
axes[0].plot(test.index, sarima_forecast.values, '--', label='SARIMA', linewidth=1.2)
axes[0].plot(test.index, hw_forecast.values, '--', label='Holt-Winters', linewidth=1.2)
axes[0].set_title('Statistical Model Forecasts', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Births')
axes[0].legend(fontsize=9)

# ML models
axes[1].plot(test_ml.index, y_test_ml.values, 'k-', linewidth=2, label='Actual', alpha=0.8)
for name, pred in ml_predictions.items():
    axes[1].plot(test_ml.index, pred, '--', label=name, linewidth=1.2)
axes[1].plot(test_ml.index, ml_avg, '-', label='ML Ensemble', linewidth=2, color='red', alpha=0.7)
axes[1].set_title('Machine Learning Model Forecasts', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Births')
axes[1].legend(fontsize=9)

# Residuals of best model
best_residuals = y_test_ml.values - ml_avg
axes[2].bar(test_ml.index, best_residuals, color='steelblue', alpha=0.6)
axes[2].axhline(0, color='red', linestyle='--')
axes[2].set_title('Forecast Residuals (ML Ensemble)', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Error (Actual - Predicted)')

plt.tight_layout()
plt.show()

## Comprehensive Results Comparison

In [ ]:
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('RMSE').reset_index(drop=True)

print("All Models Ranked by RMSE:")
print(results_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# RMSE comparison
top_n = min(10, len(results_df))
top = results_df.head(top_n)
axes[0].barh(top['Model'], top['RMSE'], color='steelblue', edgecolor='black')
axes[0].set_title('RMSE (lower is better)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('RMSE')
axes[0].invert_yaxis()

# MAE comparison
axes[1].barh(top['Model'], top['MAE'], color='coral', edgecolor='black')
axes[1].set_title('MAE (lower is better)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('MAE')
axes[1].invert_yaxis()

# MAPE comparison
axes[2].barh(top['Model'], top['MAPE'], color='#2ecc71', edgecolor='black')
axes[2].set_title('MAPE % (lower is better)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('MAPE %')
axes[2].invert_yaxis()

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Summary and Insights

**Approach Comparison**

Statistical methods (ARIMA, SARIMA, Holt-Winters) directly model the temporal structure and work well for short-term forecasting when the series has clear autocorrelation patterns. Machine learning methods (Random Forest, Gradient Boosting, XGBoost) with lagged features are more flexible and can capture non-linear relationships, but require careful feature engineering.

**Key Observations**

- The birth series is approximately stationary with mild weekly seasonality, confirmed by the ADF test and the decomposition analysis.
- SARIMA models that account for the weekly seasonal cycle tend to outperform standard ARIMA on this dataset.
- Tree-based ML models with lag and rolling features achieved competitive performance, particularly when combined as an ensemble.
- The most recent lags (lag_1 through lag_7) and rolling mean features dominate feature importance, confirming that short-term history is most informative.

**Practical Considerations**

- For operational deployment, SARIMA is straightforward to maintain and interpret, while ML ensembles offer flexibility for incorporating additional external features (weather, holidays, day type).
- The ensemble average of multiple forecasters provides a natural hedge against any single model failing.
- Forecast accuracy can be further improved by incorporating exogenous variables and extending the training window.

---

End of Notebook